In [2]:
import random
import numpy as np

from reallm import Evaluation

def evaluate_pseudorandom(func):
    randseed = random.randint(1,23489)
    try:
        res = func(randseed)
        return Evaluation(
            fitness = res,
            errors = None
        )
    except Exception as e:
        return Evaluation(
            fitness = float('-inf'),
            errors = str(e)
        )

In [3]:
from reallm import LLMSolutionEngine, Solution, ProblemSpec

## PROMPT-TO-SOLUTION ENGINE
### uses api information and problem details for easy prompt evaluation

import os

engine = LLMSolutionEngine(
    api_spec=dict(
        host="api.openai.com",
        key=os.environ['OPENAI_API_KEY'],  # your key, e.g., sk-xxxxxxxxxx
        model="gpt-4o-mini",  # your llm, e.g., gpt-3.5-turbo, deepseek-chat
        timeout=20
    ),
    problem_spec=ProblemSpec(
        description="Generate a random number (integer) from some random seed (int)",
        template="""def random_num(inp: int) -> int:\n\tpass""",
        entrypoint="random_num",
        evaluator=evaluate_pseudorandom
    )
)

In [4]:
## PROMPT SAMPLERS
### decorated string factories. can use PROBLEM_STATEMENT/PROBLEM_TEMPLATE to access problem details from engine

@engine.prompt_to_solution
def initial_solution():
    return """
<PROBLEM_DESCR>
1. First, describe your new algorithm and main steps in one sentence. The description must be inside within boxed {{}}. 
2. Next, implement the following Python function:
<PROBLEM_TEMPLATE>
Do not give additional explanations.'''
"""

@engine.prompt_to_solution
def mutated_solution(parent_solution):
    return '''<PROBLEM_DESCR>
I have one algorithm with its code as follows.
Code:
{parent_prompt}
Please assist me in creating a new algorithm that has a different form but can be a modified version of the algorithm provided.
1. First, describe your new algorithm and main steps in one sentence. The description must be inside within boxed {{}}.
2. Next, implement the following Python function:
<PROBLEM_TEMPLATE>
Do not give additional explanations.'''.format(parent_prompt=parent_solution.program)

In [8]:
@engine.prompt_to_solution.add_output(thought=r'<<(.*)>>')
def initial_solution_thinking():
    return """
<PROBLEM_DESCR>
1. First, describe your new algorithm and main steps in one sentence. The description must be inside within boxed <<>>. 
2. Next, implement the following Python function:
<PROBLEM_TEMPLATE>
Do not give additional explanations.'''
"""

yy = initial_solution_thinking()
print(yy)


solution:
def random_num(inp: int) -> int:
    a = 1664525
    c = 1013904223
    m = 2**32
    return (a * inp + c) % m


thought: This algorithm uses a linear congruential generator (LCG) to produce a random integer based on a given seed.
evaluation: Evaluation(fitness=2993210585, errors=None)


In [4]:
## MAIN LOOP
### reasons solely about the prompt composition, shouldn't need to think about prompt or LLM details at all

### useful here to know about the Solution class, intended to implement all meaningful interfaces for this flow

from itertools import chain
from typing import List

def _top_k_solutions(pop: List[Solution], k: int):
    return list(sorted(pop, key=lambda s: s.fitness, reverse=True))[:k]


def funsearch(max_gen=1, initial_pop=5, num_isles=3, max_isle_pop=3):

    all_evaluations = []

    seed_population = _top_k_solutions([initial_solution() for _ in range(initial_pop)], num_isles)

    for g_idx in range(max_gen):

        islands = []
        for is_idx, seed_soln in enumerate(seed_population):
            islands.append([mutated_solution(seed_soln) for _ in range(max_isle_pop)])

        all_solns = list(chain(*islands))
        seed_population = _top_k_solutions(all_solns, num_isles)

    return _top_k_solutions(seed_population, 1).pop()

In [5]:
%%time
roar = funsearch()

CPU times: user 552 ms, sys: 12.3 ms, total: 564 ms
Wall time: 20.1 s


In [7]:
print(roar.program)

def random_num(inp: int) -> int:
    a = 22695477
    c = 1
    m = 2**32
    
    inp = (a * inp + c) % m
    return inp ^ (inp >> 16)


